<h1>🧬 ADSP — Step 3: variants become candidate pairs</h1>

A **model** is one variant × variant pair to test for interaction. Two variants
are worth pairing when the genes they were selected for share biology — a
pathway, a disease, a protein complex.

One row of the product is a model. This is the step that decides how big the
study is: everything before it narrowed, this one multiplies.

## Why `pair_genes` and not `pair_variants`

`pair_variants` derives "this variant belongs to this gene" from coordinates.
That is right for a coding variant and wrong for a regulatory one — a branch B
variant is attached to the gene its QTL points at, which it usually does not sit
inside. `pair_variants` would look for it among that gene's positional variants,
not find it, and drop it, with no error.

`pair_genes` takes the attachment from us instead of re-deriving it: we hand it
the gene → variant mapping steps 1 and 2 produced, and it pairs the genes and
expands our list across the pairs it finds. It also owns the three rules that
are easy to get wrong — pairs are unordered, deduplication is global, and an
item on both genes of a pair must not pair with itself.

The report exists because this analysis needed it; see
[ADR-005](../../../adr/0005-pair-genes-and-caller-supplied-expansion.md).

### 1. Open a bundle, and read step 2

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from biofilter import Biofilter

bf = Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
ADSP = _root / "notebooks" / "Andre" / "adsp"
OUTPUT_DIR = ADSP / "outputs"
STEP2_PATH = OUTPUT_DIR / "step2_selected_variants.csv"

# The space our gene ids live in. Defined once because every pair_genes call in
# this notebook has to agree: a sensitivity table measured with a different
# lookup than the headline would still look plausible.
GENE_IDENTIFIER = "ensembl"

step2 = pd.read_csv(STEP2_PATH)
print(step2.branch.value_counts().to_string())
print(f"\n{step2.variant.nunique():,} variants from step 2")

[INFO] ════════════════════════════════════

[INFO] 🚀 Initializing Biofilter

[INFO]    • Version: 4.3.0

[INFO]    • Debug mode: False

[INFO]    • Config: /Users/andrerico/Works/Sys/biofilter_430/.biofilter.toml

...

[INFO]    • Views:  37 (read-only)

[INFO]    • Time:   239.8 ms

[INFO] ════════════════════════════════════

branch
B_regulatory          4982
A_protein_altering    2802


7,603 variants from step 2

### 2. The mapping: gene → the variants we chose for it

Step 2 now hands both branches the **same two identifiers**, so there is no
reconciliation left to do here. What remains is a choice of which one to use,
and it is not the obvious one.

| column | what it is for |
| --- | --- |
| `pair_gene_entity_id` | this bundle's **join key** — meaningful only next to the build that minted it |
| `pair_gene_id` | the **Ensembl id** — the portable one, and what a resolver understands |

**Use the Ensembl id.** `pair_genes` resolves its input through aliases, and an
entity id is not an alias: it is a plain number that either fails to resolve or
matches an **Entrez id belonging to a different gene**. The cell below measures
that on our own data, because it is the kind of mistake that produces a
plausible-looking wrong answer rather than an error.

Symbols are no good either — they are ambiguous, and the report warns when a
name answers to more than one gene: 111 names with symbols, 8 with Ensembl ids.

In [2]:
sample = [str(int(x)) for x in step2.pair_gene_entity_id.dropna().unique()[:400]]
probe = bf.report.run("annotate_gene", input_data=sample,
                      include_variant_summary=False).to_pandas()
found = probe[probe.entity_id.notna()]
same = found.input_value.astype(str) == found.entity_id.astype(int).astype(str)

print(f"entity ids offered as input : {len(sample)}")
print(f"  did not resolve           : {int(probe.entity_id.isna().sum())}")
print(f"  resolved to THEMSELVES    : {int(same.sum())}")
print(f"  resolved to another gene  : {int((~same).sum())}   <-- silently wrong")
found[~same][["input_value", "entity_id", "gene_symbol", "entrez_id"]].head(4)

[INFO] Report 'annotate_gene' produced 400 rows in 0.18s from bundle 9b8419b48be5004e.

entity ids offered as input : 400

  did not resolve           : 300

  resolved to THEMSELVES    : 0

  resolved to another gene  : 100   <-- silently wrong

,input_value,entity_id,gene_symbol,entrez_id
0,10811,17671.0,NOXA1,10811
1,1084,42466.0,CEACAM3,1084
3,10890,23603.0,RAB10,10890
4,10940,21214.0,POP1,10940


Every one that resolved landed on a different gene, matched through `entrez_id`.
Nothing fails — you just get pairs of the wrong genes. So the reconciliation
goes through Ensembl, using the bundle's own gene table to translate branch A.

In [3]:
mapping = step2.loc[
    step2.pair_gene_id.notna(), ["variant", "pair_gene_id", "branch"]
].rename(columns={"pair_gene_id": "ensembl_id"})
mapping["branch"] = mapping.branch.str[0]
mapping = mapping.drop_duplicates(["variant", "ensembl_id"])

unpairable = step2[step2.pair_gene_id.isna() | step2.pair_gene_entity_id.isna()]

print(f"mapping: {len(mapping):,} rows | {mapping.variant.nunique():,} variants | "
      f"{mapping.ensembl_id.nunique():,} genes")
print(f"variants on a gene this bundle has no entity for: {unpairable.variant.nunique()}")
mapping.head()

mapping: 7,784 rows | 7,603 variants | 3,094 genes

variants on a gene this bundle has no entity for: 5

,variant,ensembl_id,branch
0,1:966227:C:G,ENSG00000187583,A
1,1:973858:G:C,ENSG00000187583,A
2,1:973862:A:G,ENSG00000187583,A
3,1:973929:T:C,ENSG00000187583,A
4,1:974039:C:T,ENSG00000187583,A


### 3. One call

The mapping goes in as `gene → [items]`. The report pairs the genes, expands our
list across the pairs, and returns **item pairs as the primary table** with the
gene pairs alongside in `extra_tables`.

In [4]:
gene_to_variants = mapping.groupby("ensembl_id").variant.apply(list).to_dict()

started = time.perf_counter()
result = bf.report.run(
    "pair_genes",
    input_data=sorted(gene_to_variants),
    mapping=gene_to_variants,
    gene_identifier=GENE_IDENTIFIER,
    group_types=["Pathways"],
    max_group_size=500,
    min_group_sources=1,
)
models = result.to_pandas()
gene_pairs = result.extra_tables["gene_pairs"].to_pandas()

print(f"{len(models):,} models from {len(gene_pairs):,} gene pairs "
      f"in {time.perf_counter() - started:.1f}s")
print(f"variants appearing in a model: {len(set(models.item_1) | set(models.item_2)):,}")
models[["item_1", "item_2", "gene_1_symbol", "gene_2_symbol",
        "group_support_count", "group_support_source_count"]].head()

[WARNING] ⚠️  pair_genes: 8 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

[INFO] Report 'pair_genes' produced 817,420 rows in 0.81s from bundle 9b8419b48be5004e.

817,420 models from 100,245 gene pairs in 1.0s

variants appearing in a model: 5,420

,item_1,item_2,gene_1_symbol,gene_2_symbol,group_support_count,group_support_source_count
0,17:38752259:C:G,9:124414882:A:G,PSMB3,PSMB7,165,2
1,3:184302754:G:T,9:124414882:A:G,PSMB7,PSMD2,165,2
2,17:38752259:C:G,3:184302754:G:T,PSMB3,PSMD2,164,2
3,16:29912189:G:A,1:9650958:C:T,MAPK3,PIK3CD,90,2
4,16:29912189:G:A,1:9667109:T:C,MAPK3,PIK3CD,90,2


The warning about names answering to more than one gene is worth reading rather
than silencing: 8 of our Ensembl ids are ambiguous in this bundle, so their
variants attach to each gene the id reaches. With symbols instead it was 111.

In [5]:
print(json.dumps(result.provenance["group_filter"], indent=2))

{
  "max_group_size": 500,
  "groups_touching_input": 2526,
  "groups_kept": 2491,
  "groups_excluded_by_size": 35,
  "smallest_excluded": 521,
  "largest_seen": 2615,
  "means": "35 of the 2526 groups that reach these genes name more than 500 genes each and were excluded. The smallest one excluded has 521 genes. If the result is empty or thin, raising max_group_size is what changes it."
}

**That block is the difference between a result and a misreading.** A thin
answer here usually means the groups linking these genes were too large to
count, not that the genes share no biology.

### 4. The branch each side came from is ours to add

The item is opaque to the report — which is exactly what let us hand it a
QTL-derived attachment in the first place. The flip side is that it cannot tell
a branch A variant from a branch B one, so we label them back here.

In [6]:
branch_of = dict(zip(mapping.variant, mapping.branch))
models["branch_1"] = models.item_1.map(branch_of)
models["branch_2"] = models.item_2.map(branch_of)

print(models.groupby(["branch_1", "branch_2"]).size().to_string())

branch_1  branch_2
A         A            70777
          B           152449
B         A           139763
          B           454431

### 5. The two knobs that decide the size of the study

**`max_group_size`** drops groups naming more genes than the limit. A pathway
naming 2,615 genes links its members while saying almost nothing about any of
them.

**`min_group_sources`** requires the gene pair to be asserted by more than one
curation. It reads the source from the bundle — `group_support_source_count` —
rather than from an accession prefix, which is what an earlier draft of this
analysis had to do.

In [7]:
rows = []
for limit in (0, 500, 300, 200):
    for sources in (1, 2):
        r = bf.report.run("pair_genes", input_data=sorted(gene_to_variants),
                          mapping=gene_to_variants,
                          gene_identifier=GENE_IDENTIFIER,
                          group_types=["Pathways"],
                          max_group_size=limit, min_group_sources=sources,
                          max_pairs=50_000_000)
        rows.append({"max_group_size": "no limit" if limit == 0 else limit,
                     "min_sources": sources,
                     "gene_pairs": r.extra_tables["gene_pairs"].num_rows,
                     "models": r.num_rows})

sensitivity = pd.DataFrame(rows)
sensitivity.pivot(index="max_group_size", columns="min_sources", values="models")

[WARNING] ⚠️  pair_genes: 8 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

[INFO] Report 'pair_genes' produced 3,825,038 rows in 3.44s from bundle 9b8419b48be5004e.

[WARNING] ⚠️  pair_genes: 8 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

...

[INFO] Report 'pair_genes' produced 402,114 rows in 0.48s from bundle 9b8419b48be5004e.

[WARNING] ⚠️  pair_genes: 8 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

[INFO] Report 'pair_genes' produced 48,585 rows in 0.27s from bundle 9b8419b48be5004e.

min_sources,1,2
max_group_size,,
200,402114,48585
300,592407,66924
500,817420,91135
no limit,3825038,797453


**Every call names the same `gene_identifier`.** It happens to make no
difference on this list, and that is exactly why it is worth passing: a table
measured through a different lookup than the headline would agree today and
diverge silently the day a gene in the list has a colliding alias.

**`max_pairs` is raised here on purpose.** Its default is 1,000,000, and the
uncapped row hits it — a result that reads as a round number rather than an
answer. Any run that might exceed it should raise it or check the provenance.

**Where this lands against the target.** The meetings put the workable size at
roughly 10 million models. At `max_group_size=500` with one source we are at
~0.8M; uncapped, ~3.8M. Below budget, but within the same order of magnitude —
not the two-orders-of-magnitude headroom the partial-bundle run suggested.

### 7. A second, stricter set

The sensitivity table is for choosing; this is the choice made twice. Alongside
the headline result we ship the **two-curation** set — the same pathway cap, but
a gene pair only counts when Reactome *and* KEGG both assert it.

It is a different claim, not a smaller sample: one asks whether the genes share a
pathway, the other whether two independent curations agree that they do. Which
one the interaction testing should use is a scientific call, so both are written
and the provenance of each says which it is.

| | models | gene pairs | variants reaching a model |
| --- | ---: | ---: | ---: |
| `min_group_sources=1` | 817,420 | 100,245 | 5,420 |
| `min_group_sources=2` | 91,135 | 11,727 | 3,133 |

In [8]:
strict = bf.report.run(
    "pair_genes",
    input_data=sorted(gene_to_variants),
    mapping=gene_to_variants,
    gene_identifier=GENE_IDENTIFIER,
    group_types=["Pathways"],
    max_group_size=500,
    min_group_sources=2,
)
strict_models = strict.to_pandas()
strict_models["branch_1"] = strict_models.item_1.map(branch_of)
strict_models["branch_2"] = strict_models.item_2.map(branch_of)
strict_pairs = strict.extra_tables["gene_pairs"].to_pandas()

print(f"{len(strict_models):,} models from {len(strict_pairs):,} gene pairs")
print(f"variants appearing in a model: "
      f"{len(set(strict_models.item_1) | set(strict_models.item_2)):,}")
print(f"\nthey are a subset of the headline set: "
      f"{set(zip(strict_models.item_1, strict_models.item_2)) <= set(zip(models.item_1, models.item_2))}")

[WARNING] ⚠️  pair_genes: 8 name(s) in the mapping answer to more than one gene, so their items attach to each. Supply an entity id to say which you meant.

[WARNING] ⚠️  pair_genes: 1 gene(s) in the mapping do not resolve to a gene in this bundle, so their items cannot be paired.

[INFO] Report 'pair_genes' produced 91,135 rows in 0.36s from bundle 9b8419b48be5004e.

91,135 models from 11,727 gene pairs

variants appearing in a model: 3,133


they are a subset of the headline set: True

### 8. Export

In [9]:
product_path = OUTPUT_DIR / "step3_variant_pairs.csv"
models.rename(columns={"item_1": "variant_1", "item_2": "variant_2"}).to_csv(
    product_path, index=False)
gene_pairs.to_csv(OUTPUT_DIR / "step3_gene_pairs.csv", index=False)
sensitivity.to_csv(OUTPUT_DIR / "step3_sensitivity.csv", index=False)

Path(f"{product_path}.provenance.json").write_text(json.dumps({
    "step": "adsp_step_03_variant_pairs",
    "bundle_id": result.provenance["bundle_id"],
    "step2_input": str(STEP2_PATH),
    "reports": ["pair_genes"],
    "attachment": (
        "variant->gene came from steps 1-2 (VEP for coding, QTL target for "
        "regulatory) and was supplied to pair_genes as a mapping, not "
        "re-derived from coordinates."
    ),
    "gene_identifier": GENE_IDENTIFIER,
    "group_types": ["Pathways"],
    "max_group_size": 500,
    "min_group_sources": 1,
    "group_filter": result.provenance["group_filter"],
    "models": len(models),
}, indent=2))

# The stricter set, written beside it with its own provenance.
strict_path = OUTPUT_DIR / "step3_variant_pairs__2src.csv"
strict_models.rename(columns={"item_1": "variant_1", "item_2": "variant_2"}).to_csv(
    strict_path, index=False)
strict_pairs.to_csv(OUTPUT_DIR / "step3_gene_pairs__2src.csv", index=False)
Path(f"{strict_path}.provenance.json").write_text(json.dumps({
    "step": "adsp_step_03_variant_pairs",
    "label": "2src",
    "bundle_id": strict.provenance["bundle_id"],
    "step2_input": str(STEP2_PATH),
    "reports": ["pair_genes"],
    "gene_identifier": GENE_IDENTIFIER,
    "group_types": ["Pathways"],
    "max_group_size": 500,
    "min_group_sources": 2,
    "group_filter": strict.provenance["group_filter"],
    "models": len(strict_models),
}, indent=2))

for path in sorted(OUTPUT_DIR.glob("step3_*")):
    print(f"{path.name:<45} {path.stat().st_size / 1e6:8.2f} MB")

step3_gene_pairs.csv                             11.30 MB

step3_gene_pairs__2src.csv                        2.03 MB

step3_sensitivity.csv                             0.00 MB

step3_summary.csv                                 0.00 MB

step3_summary__2src.csv                           0.00 MB

step3_variant_pairs.csv                          96.64 MB

step3_variant_pairs.csv.provenance.json           0.00 MB

step3_variant_pairs__2src.csv                    16.74 MB

step3_variant_pairs__2src.csv.provenance.json     0.00 MB

### 9. The same thing on the command line

```bash
python notebooks/Andre/adsp/step_03_adsp_variant_pairs.py \
    --input   outputs/step2_selected_variants.csv \
    --out-dir outputs \
    --max-group-size 500 \
    --min-group-sources 2
```

### 10. Quick QA

In [10]:
checks = {
    "no variant is paired with itself": not (models.item_1 == models.item_2).any(),
    "no unordered pair appears twice":
        not models.assign(a=models[["item_1", "item_2"]].min(axis=1),
                          b=models[["item_1", "item_2"]].max(axis=1))
                  .duplicated(["a", "b"]).any(),
    "every variant came from step 2":
        bool(pd.concat([models.item_1, models.item_2]).isin(set(step2.variant)).all()),
    "the two genes of a pair are different": not (models.gene_1_id == models.gene_2_id).any(),
    "every pair has group support": bool(models.group_support_count.ge(1).all()),
    "every model has both branches labelled":
        not (models.branch_1.isna() | models.branch_2.isna()).any(),
    "the two-curation set is nested in the headline set":
        set(zip(strict_models.item_1, strict_models.item_2))
        <= set(zip(models.item_1, models.item_2)),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

paired = set(models.item_1) | set(models.item_2)
print(f"\n{len(paired):,} of {mapping.variant.nunique():,} variants ended up in a model")

PASS  no variant is paired with itself

PASS  no unordered pair appears twice

PASS  every variant came from step 2

PASS  the two genes of a pair are different

PASS  every pair has group support

PASS  every model has both branches labelled

PASS  the two-curation set is nested in the headline set


5,420 of 7,603 variants ended up in a model

### 11. Known gaps and what step 4 needs

**8 Ensembl ids in the mapping are ambiguous** in this bundle, so their variants
attach to every gene the id reaches. Small, but it inflates rather than deflates.
One further id resolves to no gene at all, and the report says so rather than
dropping it quietly — 5 variants cannot be paired.

**A gene's variants pair only if the partner gene also carries some.** Variants
whose gene has no partner in our set never appear in a model — that is the
report's both-sides rule, not a bug.

**`group_support_count` is a weight, not a p-value.** It counts groups under the
size limit chosen, so it moves with `max_group_size`.

**Step 4 is the interaction testing itself**, which happens outside Biofilter.
The product here is its input.